In [11]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

model_data = pd.read_csv("data/processed/test_training_data.csv")
print(model_data.shape)
model_data.head()

(1963, 18)


,"last_name, first_name",player_id,year,hit,single,double,triple,home_run,strikeout,walk,b_rbi,b_total_bases,r_total_stolen_base,r_run,season,fantasy_points,next_season_points,next_season
0,"Abrams, CJ",682928,2022,70,54,12,2,2,50,5,21,92,7,33,2022,108,340.0,2023.0
1,"Abrams, CJ",682928,2023,138,86,28,6,18,118,32,64,232,47,83,2023,340,321.0,2024.0
2,"Abrams, CJ",682928,2024,133,78,29,6,20,128,40,65,234,31,79,2024,321,346.0,2025.0
3,"Abreu, José",547989,2015,178,111,34,3,30,140,39,101,308,0,88,2015,396,381.0,2016.0
4,"Abreu, José",547989,2016,183,125,32,1,25,125,47,100,292,0,67,2016,381,459.0,2017.0


In [3]:
model_data.isnull().sum() #find empty data

last_name, first_name    0
player_id                0
year                     0
hit                      0
single                   0
double                   0
triple                   0
home_run                 0
strikeout                0
walk                     0
b_rbi                    0
b_total_bases            0
r_total_stolen_base      0
r_run                    0
season                   0
fantasy_points           0
next_season_points       0
next_season              0
dtype: int64

In [4]:
model_data["gap"] = model_data["next_season"] - model_data["year"]
model_data["gap"].value_counts()

gap
1.0    1625
2.0     268
3.0      53
4.0      11
6.0       3
5.0       3
Name: count, dtype: int64

In [8]:
model_data = model_data[model_data["gap"] == 1.0].copy() #remove year gaps (ex: 2021 - 2023)
print(model_data.shape)
model_data["gap"].value_counts()

(1625, 19)


gap
1.0    1625
Name: count, dtype: int64

In [9]:
features = [
    "single",
    "double", 
    "triple",
    "home_run",
    "strikeout",
    "walk",
    "b_rbi",
    "r_total_stolen_base",
    "r_run",
    "fantasy_points"
]
target = "next_season_points"

x = model_data[features]
y = model_data[target]

print(x.shape)
print(y.describe())

(1625, 10)
count    1625.000000
mean      281.920615
std       108.665848
min        58.000000
25%       197.000000
50%       270.000000
75%       356.000000
max       707.000000
Name: next_season_points, dtype: float64


In [10]:
train = model_data[model_data["year"] < 2024] #data used for traning
test = model_data[model_data["year"] == 2024] #data used for testing (precict 2025 points)

x_train = train[features] #current season points (pre-2024)
x_test = test[features]
y_train = train[target] #next season points (pre-2024)
y_test = test[target]

print(f"training row: {len(train)}")
print(f"testing row: {len(test)}")

training row: 1426
testing row: 199


In [12]:
model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
model.fit(x_train, y_train)

y_predict = model.predict(x_test)

mae = mean_absolute_error(y_test, y_predict)
print(f"Mean Absolute Error: {mae:.1f} fantasy points")

Mean Absolute Error: 70.1 fantasy points


In [15]:
result1 = test[["last_name, first_name", "year", "fantasy_points"]].copy()
result1["predicted"] = y_predict.round(1)
result1["actual"] = y_test.values
result1["error"] = (result1["predicted"] - result1["actual"]).abs()
result1.sort_values("error", ascending=False).head(10)

,"last_name, first_name",year,fantasy_points,predicted,actual,error
1435,"Raleigh, Cal",2024,311,275.899994,509.0,233.100006
1350,"Perdomo, Geraldo",2024,211,284.399994,512.0,227.600006
938,"Judge, Aaron",2024,630,374.299988,599.0,224.700012
1519,"Rodríguez, Julio",2024,282,220.899994,432.0,211.100006
184,"Bichette, Bo",2024,121,206.399994,406.0,199.600006
1751,"Tatis Jr., Fernando",2024,256,247.199997,439.0,191.800003
209,"Bohm, Alec",2024,366,435.700012,251.0,184.700012
1289,"Ohtani, Shohei",2024,653,388.000000,570.0,182.000000
49,"Alonso, Pete",2024,359,258.899994,440.0,181.100006
1658,"Soler, Jorge",2024,294,281.500000,107.0,174.500000
